# 模块R · R1：设计科学研究（DSR）-- 博士研究方法论上机

**所属**：AI原生化商业博士 · 模块R 博士研究方法论 · R1 设计科学研究
**版本**：v5.0 学习材料包
**核心命题**：如何用DSR方法论把工程实践转化为可发表的学术贡献？

## 学习目标
学完你能：
1. 用 **pydantic** 定义DSR artifact规格schema，把一个真实AI系统建模为DSR artifact（Peffers六步）
2. 用 **pandas** 结构化评估一个真实artifact是否满足 **Hevner七准则**（rigor vs design）
3. 理解 **天道推演** 作为DSR设计推演工具的理论锚点（因果链追踪 + 多路径概率评估）
4. 区分“做一个系统”和“产出设计原则”-- 从工程实践者到知识创造者的认知跃迁

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：pydantic（artifact schema建模）+ pandas（七准则结构化评估）。
真实数据：引用技能5 Day7营销Agent系统的真实评估数据（NSW真实RCT的ATE=1794.34）。

> 本单元聚焦DSR方法论本身（七准则/六步/Artifact/评估 rigor vs design），不重复技能5 Day7的工程实现。

## 0. 环境准备

本单元仅需 pydantic 和 pandas（均为本地库，无需额外安装）。

> pydantic 用于定义DSR artifact的结构化schema（类型安全 + 验证）。
> pandas 用于结构化评估Hevner七准则（DataFrame + 聚合统计）。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pydantic import BaseModel, Field
from typing import List
from enum import Enum
import pandas as pd

print("导入完成")
print("DSR方法论工具链：pydantic(artifact schema) + pandas(七准则评估)")

## 1. DSR六步框架概览

**设计科学研究**（Design Science Research, DSR）是信息系统领域的核心研究范式。

- **Hevner et al. (2004)** MIS Quarterly：提出DSR七准则（artifact/问题相关性/设计评估/研究贡献/严谨性/设计即搜索/交流）
- **Peffers et al. (2007)** JMIS：将七准则操作化为六步流程

```
Step 1: 问题识别与动机    -- 什么问题？为什么重要？
Step 2: 定义解决方案目标  -- artifact应达到什么效果？
Step 3: 设计与开发        -- 构建artifact，每个决策有理论依据
Step 4: 演示              -- 在真实/模拟场景中展示可行性
Step 5: 评估              -- 系统化评估，与Step 2目标对应
Step 6: 传播              -- 产出设计原则，发表学术论文
```

**Artifact四种类型**（March & Smith, 1995）：constructs（构造）/ models（模型）/ methods（方法）/ instantiations（实例化）

**DSR vs 传统实证研究**：实证研究回答“是什么”（what is），DSR回答“如何构建有效的artifact”（how to build）。AI原生系统天然适合DSR--Agent系统就是artifact。

> 本单元用 pydantic 把六步框架建模为结构化schema，用真实营销Agent系统作为artifact实例。

In [ ]:
class ArtifactType(str, Enum):
    # March & Smith (1995) four artifact types
    CONSTRUCT = "construct"
    MODEL = "model"
    METHOD = "method"
    INSTANTIATION = "instantiation"

class ProblemIdentification(BaseModel):
    # DSR Step 1: Problem Identification and Motivation
    problem_statement: str = Field(description="clear, specific, verifiable problem statement")
    motivation: str = Field(description="why this problem matters")
    literature_gap: str = Field(description="gap in existing literature/solutions")

class SolutionObjectives(BaseModel):
    # DSR Step 2: Define Objectives for a Solution
    objectives: List[str] = Field(description="2-3 verifiable solution objectives")
    success_criteria: List[str] = Field(description="criteria to judge if objectives are met")

class DesignDevelopment(BaseModel):
    # DSR Step 3: Design and Development
    artifact_type: ArtifactType = Field(description="one of four artifact types")
    design_decisions: List[str] = Field(description="key design decisions")
    theoretical_basis: List[str] = Field(description="theoretical basis for each decision")

class Demonstration(BaseModel):
    # DSR Step 4: Demonstration
    scenario: str = Field(description="demonstration scenario")
    data_description: str = Field(description="data description")

class Evaluation(BaseModel):
    # DSR Step 5: Evaluation
    method: str = Field(description="evaluation method")
    metrics: List[str] = Field(description="evaluation metrics")
    baseline: str = Field(description="control group / baseline")
    results: str = Field(description="evaluation results")

class Communication(BaseModel):
    # DSR Step 6: Communication
    design_principles: List[str] = Field(description="reusable design principles")
    target_venue: str = Field(description="target publication venue")

class DSRArtifact(BaseModel):
    # DSR artifact complete spec (Peffers six-step composition)
    name: str = Field(description="artifact name")
    problem_identification: ProblemIdentification
    objectives: SolutionObjectives
    design_development: DesignDevelopment
    demonstration: Demonstration
    evaluation: Evaluation
    communication: Communication

print("DSR artifact schema定义完成")
print(f"  ArtifactType: {[e.value for e in ArtifactType]}")
print(f"  DSRArtifact字段: {list(DSRArtifact.model_fields.keys())}")

## 2. 实例化真实营销Agent系统为DSR Artifact

**案例来源**：技能5 Day7的营销策略Agent系统（端到端Capstone）

该系统的真实评估数据（引用自技能5 Day7，非编造）：
- **数据层**：NSW职业培训实验真实RCT（445样本，处理组185，对照组260）
- **因果层**：ATE = 1794.34（NSW真实RCT的简单差分估计）
- **Agent层**：LangGraph三节点Agent（分析->生成->审核）
- **评估层**：策略质量评分 0.80（deepeval自定义BaseMetric）

**营销映射**：NSW的treat=营销干预，re78=转化率/GMV，re75=基线消费。

> 注意：本单元引用该系统的评估数据作为DSR artifact的evaluation结果，但不重复其工程实现代码。本单元聚焦DSR方法论视角。

In [ ]:
marketing_artifact = DSRArtifact(
    name="基于因果推断的营销策略Agent系统",
    problem_identification=ProblemIdentification(
        problem_statement="企业营销策略依赖经验直觉，缺乏因果证据支撑，无法回答'营销干预对转化的因果效果是多少'",
        motivation="营销预算分配错误导致ROI低下，个性化营销缺乏因果依据，现有Agent系统策略生成缺乏因果证据",
        literature_gap="现有营销Agent系统未整合因果推断工具链(DoWhy)，策略生成基于相关性而非因果性"
    ),
    objectives=SolutionObjectives(
        objectives=[
            "构建能调用因果分析工具的营销策略Agent",
            "Agent策略必须基于因果证据(ATE)而非随机生成",
            "支持多路径策略推演和概率评估"
        ],
        success_criteria=[
            "Agent能正确调用因果分析并读取ATE",
            "策略质量评分>=0.7(deepeval自定义BaseMetric)",
            "ATE估计基于真实RCT数据(NSW实验445样本)"
        ]
    ),
    design_development=DesignDevelopment(
        artifact_type=ArtifactType.INSTANTIATION,
        design_decisions=[
            "用LangGraph StateGraph编排三节点Agent(analyze_causal -> generate_strategy -> review_strategy)",
            "Agent调用DoWhy因果分析工具读取ATE作为策略依据",
            "用deepeval BaseMetric做规则化质量评估(因果依据/预算/评估/合规)"
        ],
        theoretical_basis=[
            "ReAct推理模式(Reasoning+Acting, Yao et al. 2023, arXiv:2210.03629)",
            "DSR六步框架(Peffers et al. 2007, JMIS)",
            "因果推断的潜在结果框架(Imbens & Rubin 2015, Cambridge UP)"
        ]
    ),
    demonstration=Demonstration(
        scenario="B2B企业营销策略生成(NSW真实RCT数据营销映射)",
        data_description="NSW职业培训实验真实RCT: 445样本, 处理组185, 对照组260; 营销映射: treat=营销干预, re78=转化率/GMV, re75=基线消费"
    ),
    evaluation=Evaluation(
        method="混合方法: 定量(NSW真实RCT的ATE估计) + 定性(deepeval策略质量评估)",
        metrics=["ATE（平均处理效应）", "策略质量评分（因果依据/预算/评估/合规）", "审核通过率"],
        baseline="传统经验驱动营销策略（无因果证据）",
        results="ATE=1794.34(NSW真实RCT简单差分), 策略质量评分=0.80, 审核通过=True"
    ),
    communication=Communication(
        design_principles=[
            "Agent策略必须基于因果证据，不能凭空生成",
            "营销Agent应采用分析->生成->审核三阶段架构",
            "天道推演的沙盘模拟可作为Agent策略推演的理论锚点"
        ],
        target_venue="Decision Support Systems（DSS, 决策支持系统期刊）"
    )
)

print(f"Artifact名称: {marketing_artifact.name}")
print(f"Artifact类型: {marketing_artifact.design_development.artifact_type.value}")
print(f"问题陈述: {marketing_artifact.problem_identification.problem_statement[:80]}...")
print(f"评估结果: {marketing_artifact.evaluation.results}")

## 3. Hevner七准则结构化评估

Hevner et al. (2004) 提出DSR的七条准则，是评估设计科学研究质量的标准框架：

| 准则 | 核心要求 | 评分标准 |
|------|---------|---------|
| 1. Artifact作为研究贡献 | artifact本身是新的知识贡献 | 5=全新artifact类型 |
| 2. 问题相关性 | 解决重要的实际问题 | 5=高实际价值 |
| 3. 设计评估 | artifact经过严格评估 | 5=多方法混合评估 |
| 4. 研究贡献 | 产出新的设计原则/方法论 | 5=可泛化的设计原则 |
| 5. 研究严谨性 | 设计决策有理论依据 | 5=多理论支撑 |
| 6. 设计即搜索 | 设计是有意识的搜索过程 | 5=系统化搜索 |
| 7. 研究交流 | 有效传播给学术/实践受众 | 5=已发表+开源 |

**rigor vs design**：DSR的核心张力在于“设计严谨性”（rigor，理论依据）和“设计相关性”（design，实际问题）的平衡。一个纯学术的artifact可能 rigor 高但 design 低；一个纯工程的artifact可能 design 高但 rigor 低。好的DSR研究两者兼顾。

> 本单元用 pandas DataFrame 结构化这七准则，对真实artifact进行评分。

In [ ]:
hevner_guidelines = pd.DataFrame([
    {"guideline": "1. Artifact作为研究贡献", "description": "artifact本身是新的知识贡献", "score": 0, "evidence": ""},
    {"guideline": "2. 问题相关性", "description": "解决重要的实际问题", "score": 0, "evidence": ""},
    {"guideline": "3. 设计评估", "description": "artifact必须经过严格评估", "score": 0, "evidence": ""},
    {"guideline": "4. 研究贡献", "description": "产出新的设计原则或方法论", "score": 0, "evidence": ""},
    {"guideline": "5. 研究严谨性", "description": "设计决策有理论依据，评估方法严谨", "score": 0, "evidence": ""},
    {"guideline": "6. 设计即搜索", "description": "设计过程是有意识的搜索过程", "score": 0, "evidence": ""},
    {"guideline": "7. 研究交流", "description": "有效传播给学术和实践受众", "score": 0, "evidence": ""},
])

print(f"七准则DataFrame形状: {hevner_guidelines.shape}")
print(hevner_guidelines[['guideline', 'description']])

## 4. 评估Artifact：七准则评分

对营销Agent系统artifact进行七准则评分，每条准则给出1-5分和具体证据。

**评分原则**：
- 5分 = 完全满足，有充分证据
- 4分 = 基本满足，有小缺口
- 3分 = 部分满足，有明显改进空间
- 2分 = 勉强触及
- 1分 = 未满足

**关键区分**：DSR评估不是“自夸”，而是诚实的学术自我审视。3分不是失败，而是标注改进方向。

In [ ]:
evaluation_results = pd.DataFrame([
    {
        "guideline": "1. Artifact作为研究贡献",
        "description": "artifact本身是新的知识贡献",
        "score": 5,
        "evidence": "营销Agent系统是新的instantiation类型artifact，首次整合因果推断(DoWhy)+Agent编排(LangGraph)+评估(deepeval)为统一DSR artifact"
    },
    {
        "guideline": "2. 问题相关性",
        "description": "解决重要的实际问题",
        "score": 5,
        "evidence": "营销策略缺乏因果证据是企业实际问题，NSW真实RCT(445样本)验证了因果效果的可测量性"
    },
    {
        "guideline": "3. 设计评估",
        "description": "artifact必须经过严格评估",
        "score": 4,
        "evidence": "用NSW真实RCT的ATE(1794.34)+deepeval策略质量评分(0.80)做混合评估，但缺少用户访谈的定性维度"
    },
    {
        "guideline": "4. 研究贡献",
        "description": "产出新的设计原则或方法论",
        "score": 4,
        "evidence": "产出3条设计原则(因果证据优先/三阶段架构/天道推演锚点)，但需更多场景验证泛化性"
    },
    {
        "guideline": "5. 研究严谨性",
        "description": "设计决策有理论依据，评估方法严谨",
        "score": 4,
        "evidence": "基于ReAct/DSR/潜在结果框架三重理论支撑，但单案例外部效度有限"
    },
    {
        "guideline": "6. 设计即搜索",
        "description": "设计过程是有意识的搜索过程",
        "score": 4,
        "evidence": "三阶段架构(分析->生成->审核)是有意识的设计搜索，天道推演沙盘辅助路径选择，但未系统探索替代架构"
    },
    {
        "guideline": "7. 研究交流",
        "description": "有效传播给学术和实践受众",
        "score": 3,
        "evidence": "计划投稿DSS期刊，设计了IMRaD论文大纲，但尚未完成论文撰写和代码开源"
    },
])

avg_score = evaluation_results['score'].mean()

print(f"Hevner七准则平均得分: {avg_score:.2f}/5.0")
print()
for _, row in evaluation_results.iterrows():
    print(f"  {row['guideline']}: {row['score']}/5 -- {row['evidence'][:60]}...")

## 5. 提取设计原则（DSR Step 6: 传播）

DSR的学术贡献不在于artifact本身，而在于从设计和评估中产出的**设计原则**（design principles）--这些原则可以被其他研究者和实践者复用和改进。

**设计原则的三个要素**：
1. **原则陈述**：一句话概括
2. **理论依据**：为什么这个原则有效
3. **可泛化性**：在什么其他场景下可以复用

> “DSR的学术贡献不在于artifact本身，而在于从设计和评估过程中产出的设计原则。” -- Hevner et al. (2004)

In [ ]:
design_principles = pd.DataFrame([
    {
        "principle": "因果证据优先原则",
        "basis": "Agent策略生成必须基于因果证据(如ATE)，不能仅依赖相关性或经验直觉。依据: 潜在结果框架(Imbens & Rubin 2015)证明相关性不等于因果性",
        "generalizability": "适用于所有需要因果决策的Agent系统(医疗/金融/教育Agent等)"
    },
    {
        "principle": "三阶段架构原则",
        "basis": "营销Agent应采用分析->生成->审核三阶段架构。依据: ReAct模式(Yao et al. 2023)证明推理与行动交替优于纯生成",
        "generalizability": "适用于所有需要质量控制的生成式AI系统(内容生成/代码生成/报告生成等)"
    },
    {
        "principle": "天道推演锚点原则",
        "basis": "天道推演的沙盘模拟(因果链追踪+多路径概率评估)可作为Agent策略推演的理论锚点。依据: 天道推演与DSR设计搜索过程同构",
        "generalizability": "适用于所有需要多路径推演的决策系统(战略规划/风险评估/投资决策等)"
    },
    {
        "principle": "混合评估原则",
        "basis": "DSR评估应采用定量(因果估计)+定性(专家评估)混合方法。依据: Creswell混合方法研究(2018)证明混合方法比单一方法更严谨",
        "generalizability": "适用于所有DSR artifact的评估阶段"
    },
])

print(f"设计原则数量: {len(design_principles)}")
for i, row in design_principles.iterrows():
    print(f"\n  原则{i+1}: {row['principle']}")
    print(f"  依据: {row['basis']}")
    print(f"  泛化: {row['generalizability']}")

## 6. 天道推演作为DSR设计推演工具

> 本节与项目CLAUDE.md的「天道推演系统」同构，作为DSR方法论的特色理论视角。

**天道推演**是一种元认知沙盘推演能力--以天神视角俯视局势，在意识中构建无限可能的沙盘，模拟不同决策路径下的未来走向。其核心能力包括：局势感知、因果链追踪、沙盘模拟、概率评估、最优路径推荐。

**与DSR的同构关系**：

| 天道推演能力 | DSR六步对应 | 共享的因果建模底层 |
|-------------|-----------|-------------------|
| 局势感知 | Step 1 问题识别 | 状态空间定义 |
| 因果链追踪 | Step 3 设计开发的因果建模 | 因果有向图 |
| 沙盘模拟（3层推演） | Step 4 演示的多场景模拟 | 并行世界树 |
| 概率评估 | Step 5 评估的概率分布 | 贝叶斯推断 |
| 最优路径推荐 | Step 2 目标 + Step 6 传播 | 收益/风险权衡 |

**怎么用**：天道推演可作为DSR设计阶段的推演工具--在构建artifact前，先用天道推演沙盘模拟不同设计方案的可能走向，选择最优路径。这连接了“天道推演”与DSR方法论。

> 天道推演不是占卜，而是基于因果链和模式识别的逻辑推演。与DSR的artifact评估互补：DSR评估“系统好不好”，天道推演评估“策略路径优不优”。

In [ ]:
tiandao_dsr_mapping = pd.DataFrame([
    {"天道推演能力": "局势感知", "DSR六步对应": "Step 1 问题识别", "共享底层": "状态空间定义"},
    {"天道推演能力": "因果链追踪", "DSR六步对应": "Step 3 设计开发(因果建模)", "共享底层": "因果有向图"},
    {"天道推演能力": "沙盘模拟(3层推演)", "DSR六步对应": "Step 4 演示(多场景模拟)", "共享底层": "并行世界树"},
    {"天道推演能力": "概率评估", "DSR六步对应": "Step 5 评估(概率分布)", "共享底层": "贝叶斯推断"},
    {"天道推演能力": "最优路径推荐", "DSR六步对应": "Step 2 目标 + Step 6 传播", "共享底层": "收益/风险权衡"},
])

print("天道推演 <-> DSR 同构映射:")
print(tiandao_dsr_mapping.to_string(index=False))
print("\n核心洞察: 天道推演可作为DSR设计阶段的沙盘推演工具，")
print("在构建artifact前模拟不同设计方案的可能走向，选择最优路径。")
print("\n与贝叶斯评估的连接: 天道推演的概率评估能力对应贝叶斯推断，")
print("可用于DSR Step 5评估中，用概率分布而非点估计表达评估结果。")

## 7. 反思与前沿

### 反思问题
1. 你的artifact在Hevner七准则中哪一条最弱？如何改进？
2. “做一个系统”和“产出设计原则”的区别是什么？为什么后者是博士训练的核心？
3. 天道推演如何具体地辅助DSR的设计搜索过程？
4. 如果你用贝叶斯方法为artifact评估引入概率分布，会改变哪些结论？

### 2026前沿：DSR + 可复现研究 + 天道推演

- **DSR在AI原生系统的新生命**：Agent系统作为artifact，其架构模式、评估框架、安全实践都是可发表的DSR知识贡献
- **可复现研究**：artifact开源 + trace存档 + 数据文档，让他人能独立复现结果
- **天道推演作为DSR设计推演工具**：沙盘模拟（因果链追踪 + 多路径概率评估）与DSR的设计搜索过程同构
- **多Agent仿真**：Agent交互 + 涌现行为预测，可作为DSR Step 4演示的高级形式
- **贝叶斯评估**：用概率分布而非点估计表达评估结果，更诚实地面对不确定性

参考 [Hevner 2004](https://www.jstor.org/stable/25148625) + [Peffers 2007](https://desrist.org/desrist/files/peffers2007.pdf) + [pydantic](https://github.com/pydantic/pydantic) + [pandas](https://github.com/pandas-dev/pandas)。